In [2]:
!pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers accelerate

Looking in indexes: https://download.pytorch.org/whl/cu121


In [15]:
import pandas as pd
import torch
import os
import re
from sentence_transformers import SentenceTransformer

COMBINED_CSV = "all_results.csv"
ROUTER_CSV_OUT = "router_data.csv"
CHECKPOINT_OUT = "router_data_checkpoint.pt"

def create_router_dataset_and_checkpoint():

    if not os.path.exists(COMBINED_CSV):
        print(f"Input file not found: {COMBINED_CSV}")
        return

    print(f"Reading {COMBINED_CSV}...")
    df = pd.read_csv(COMBINED_CSV, low_memory=False)

    if "TaskID" in df.columns:
        df["Question"] = df["Question"].fillna(df["TaskID"])

    if "Benchmark" not in df.columns:
        df["Benchmark"] = "unknown"

    if "Reward" not in df.columns:
        raise RuntimeError("Reward column not found")

    df["Reward"] = pd.to_numeric(df["Reward"], errors="coerce")
    df = df.dropna(subset=["Question", "Benchmark", "Reward"])

    print("Pivoting rewards...")

    pivot = df.pivot_table(
        index=["Question", "Benchmark"],
        columns="Model",
        values="Reward",
        aggfunc="mean"
    )

    def natural_sort_key(s):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

    pivot = pivot[sorted(pivot.columns, key=natural_sort_key)]
    pivot = pivot.fillna(pivot.min().min())

    pivot.reset_index(inplace=True)
    pivot.rename(columns={"Question": "question_text", "Benchmark": "benchmark"}, inplace=True)

    pivot.to_csv(ROUTER_CSV_OUT, index=False)

    print(f"Router CSV saved: {pivot.shape}")

    texts = pivot["question_text"].astype(str).tolist()
    benchmarks = pivot["benchmark"].tolist()

    model_cols = sorted([c for c in pivot.columns if c.startswith("model_")], key=natural_sort_key)

    Y_rewards = torch.tensor(pivot[model_cols].values, dtype=torch.float32)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading embedder on {device}")

    embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)

    print(f"Encoding {len(texts)} questions...")
    X = embedder.encode(texts, convert_to_tensor=True, show_progress_bar=True).cpu()

    torch.save(
        {
            "X": X,
            "Y_rewards": Y_rewards,
            "texts": texts,
            "benchmarks": benchmarks,
            "model_names": model_cols,
        },
        CHECKPOINT_OUT,
    )

    print("Checkpoint saved")
    print("X:", X.shape)
    print("Y:", Y_rewards.shape)

create_router_dataset_and_checkpoint()

Reading all_results.csv...
Pivoting rewards...
Router CSV saved: (17590, 32)
Loading embedder on cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 17590 questions...


Batches:   0%|          | 0/550 [00:00<?, ?it/s]

Checkpoint saved
X: torch.Size([17590, 1024])
Y: torch.Size([17590, 30])


In [16]:
# actually training the router

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import os
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import Normalizer
import torch.nn.functional as F

# ===================== USER CONFIG =====================
SEED = 42
CHECKPOINT_FILE = os.path.join("router_data_checkpoint.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TOPK = 3
EPOCHS = 50

BATCH_SIZE_TRAIN = 64
BATCH_SIZE_EVAL = 256

LR = 1e-3

# Softmax temperature schedule (explore -> commit)
TAU_START = 2.0
TAU_END   = 0.3

# Entropy bonus schedule (encourage exploration early, off later)
ENT_BONUS_START = 0.10   # subtract this * entropy from loss
ENT_BONUS_END   = 0.0

# Auxiliary CE-to-oracle head weight schedule
CE_W_START = 0.30
CE_W_END   = 0.05

# Hard-example reweighting
USE_HARD_REWEIGHT = True
HARD_REWEIGHT_POWER = 1.0   # 1.0 = linear, >1 emphasizes harder examples
HARD_REWEIGHT_CLIP = 10.0   # cap weights to avoid instability

# Warmup: train full expected-reward first, then switch to top-K-aligned loss
WARMUP_EPOCHS = 5
# ======================================================


# ------------------ SEED ------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ------------------ LOAD DATA ------------------
print(f"Loading data from {CHECKPOINT_FILE}")
data = torch.load(CHECKPOINT_FILE)

X = data["X"].numpy()

for k in ["Y_rewards", "Y", "rewards", "R"]:
    if k in data:
        Y = data[k].float()
        break
else:
    raise KeyError("No reward matrix found in checkpoint (looked for Y_rewards/Y/rewards/R).")

num_experts = Y.shape[1]
print(f"Found reward matrix with shape: {tuple(Y.shape)} (N, num_experts)")

# ------------------ DEDUPLICATION ------------------
print(">>> Removing duplicate (X, Y) pairs")

X_np = X.copy()
Y_np = Y.numpy()

# Create a hashable representation by concatenation
XY = np.concatenate([X_np, Y_np], axis=1)

# Find unique rows
_, unique_idx = np.unique(XY, axis=0, return_index=True)
unique_idx = np.sort(unique_idx)

X = X_np[unique_idx]
Y = torch.tensor(Y_np[unique_idx], dtype=torch.float32)

print(f"Deduplicated dataset size: {len(X)}")
print(f"Removed {len(X_np) - len(X)} duplicates")

# ------------------ NORMALIZE ------------------
scaler = Normalizer(norm="l2")
X = scaler.fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)

# ------------------ SPLIT ------------------
N = len(X)
perm = torch.randperm(N)

train_idx = perm[: int(0.8 * N)]
val_idx   = perm[int(0.8 * N) : int(0.9 * N)]
test_idx  = perm[int(0.9 * N) :]

X_train, Y_train = X[train_idx], Y[train_idx]
X_val,   Y_val   = X[val_idx],   Y[val_idx]
X_test,  Y_test  = X[test_idx],  Y[test_idx]

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=BATCH_SIZE_TRAIN, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, Y_val), batch_size=BATCH_SIZE_EVAL)
test_loader  = DataLoader(TensorDataset(X_test, Y_test), batch_size=BATCH_SIZE_EVAL)

# ------------------ DATASET STATS ------------------
print("\n=== DATASET STATS ===")
print(f"Total datapoints:      {N}")
print(f"Train datapoints:      {len(train_idx)}")
print(f"Validation datapoints: {len(val_idx)}")
print(f"Test datapoints:       {len(test_idx)}")
print(f"Number of experts:     {num_experts}")
print("=====================\n")

# ------------------ BASELINES ------------------
best_single_idx = Y_train.mean(dim=0).argmax().item()

best_single_val_reward = Y_val[:, best_single_idx].mean().item()
best_single_test_reward = Y_test[:, best_single_idx].mean().item()

oracle_val_reward = Y_val.max(dim=1).values.mean().item()
oracle_test_reward = Y_test.max(dim=1).values.mean().item()

print("\n=== BASELINES ===")
print(f"Best single expert idx:  {best_single_idx}")
print(f"Best single VAL reward:  {best_single_val_reward:.4f}")
print(f"Best single TEST reward: {best_single_test_reward:.4f}")
print(f"Oracle VAL reward:       {oracle_val_reward:.4f}")
print(f"Oracle TEST reward:      {oracle_test_reward:.4f}")
print("=================\n")

# ------------------ MODEL ------------------
class ResidualRouter(nn.Module):
    """
    Slightly higher-capacity router than TinyRouter.
    Residual MLP keeps optimization stable while adding nonlinearity.
    """
    def __init__(self, input_dim: int, num_experts: int, hidden: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, num_experts)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h))
        h = h + h2
        return self.out(h)

router = ResidualRouter(X.shape[1], num_experts).to(DEVICE)

# ------------------ SCHEDULE HELPERS ------------------
def lerp(a: float, b: float, t: float) -> float:
    return a + (b - a) * t

def exp_interp(a: float, b: float, t: float) -> float:
    # exponential interpolation a -> b
    if a <= 0 or b <= 0:
        return lerp(a, b, t)
    return a * (b / a) ** t

# ------------------ LOSSES ------------------
def expected_reward_per_sample(logits, rewards, tau: float):
    probs = F.softmax(logits / tau, dim=1)
    exp_reward = (probs * rewards).sum(dim=1)
    return -exp_reward  # shape [B]

def topk_expected_reward_per_sample(logits, rewards, k: int, tau: float):
    probs = F.softmax(logits / tau, dim=1)
    topk_vals, topk_idx = probs.topk(k, dim=1)
    topk_vals = topk_vals / (topk_vals.sum(dim=1, keepdim=True) + 1e-12)
    topk_rewards = rewards.gather(1, topk_idx)
    exp_reward = (topk_vals * topk_rewards).sum(dim=1)
    return -exp_reward  # shape [B]

def entropy_mean(logits, tau: float = 1.0):
    probs = F.softmax(logits / tau, dim=1)
    return -(probs * (probs.clamp_min(1e-12)).log()).sum(dim=1).mean()

# ------------------ OPTIM ------------------
optimizer = optim.Adam(router.parameters(), lr=LR)

# ------------------ TRAIN ------------------
print(">>> Training router (top-K aligned + temperature anneal + entropy schedule + oracle-CE aux + optional hard reweight)")

for epoch in range(EPOCHS):
    router.train()

    t = epoch / max(1, (EPOCHS - 1))
    tau = exp_interp(TAU_START, TAU_END, t)
    ent_bonus = lerp(ENT_BONUS_START, ENT_BONUS_END, t)
    ce_w = lerp(CE_W_START, CE_W_END, t)

    use_topk_loss = (epoch >= WARMUP_EPOCHS)

    running_loss = 0.0
    running_reward = 0.0
    n_seen = 0

    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)

        optimizer.zero_grad()
        logits = router(bx)

        # Reward loss (either full softmax ER or top-K ER)
        if use_topk_loss:
            per_sample = topk_expected_reward_per_sample(logits, by, TOPK, tau)
        else:
            per_sample = expected_reward_per_sample(logits, by, tau)

        # Optional hard-example reweighting: weight samples by (oracle - current)^power
        if USE_HARD_REWEIGHT:
            with torch.no_grad():
                oracle = by.max(dim=1).values
                probs_full = F.softmax(logits / tau, dim=1)
                current = (probs_full * by).sum(dim=1)
                gap = (oracle - current).clamp(min=0.0)
                w = (gap + 1e-6) ** HARD_REWEIGHT_POWER
                w = w / (w.mean() + 1e-12)
                w = w.clamp(max=HARD_REWEIGHT_CLIP)
            reward_loss = (w * per_sample).mean()
        else:
            reward_loss = per_sample.mean()

        # Auxiliary classification to oracle argmax (improves routing boundaries)
        oracle_idx = by.argmax(dim=1)
        ce_loss = F.cross_entropy(logits, oracle_idx)

        # Entropy bonus (encourage exploration early; turn off later)
        ent = entropy_mean(logits, tau=1.0)

        loss = reward_loss + ce_w * ce_loss - ent_bonus * ent
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            probs_eval = F.softmax(logits / tau, dim=1)
            exp_reward_batch = (probs_eval * by).sum(dim=1).mean().item()

        running_loss += loss.item() * bx.size(0)
        running_reward += exp_reward_batch * bx.size(0)
        n_seen += bx.size(0)

    # ------------------ VALIDATION EXPECTED REWARD ------------------
    router.eval()
    val_total, val_n = 0.0, 0
    with torch.no_grad():
        for vx, vy in val_loader:
            vx, vy = vx.to(DEVICE), vy.to(DEVICE)
            probs = F.softmax(router(vx) / tau, dim=1)
            val_total += (probs * vy).sum(dim=1).sum().item()
            val_n += vx.size(0)

    # Routing entropy (diagnostic)
    router_ent = 0.0
    ent_n = 0
    with torch.no_grad():
        for vx, _ in val_loader:
            vx = vx.to(DEVICE)
            p = F.softmax(router(vx), dim=1)
            router_ent += (-(p * (p.clamp_min(1e-12)).log()).sum(dim=1)).sum().item()
            ent_n += vx.size(0)
    router_ent /= max(1, ent_n)

    print(
        f"Epoch {epoch:02d} | "
        f"tau={tau:.3f} | topk_loss={'Y' if use_topk_loss else 'N'} | "
        f"trainLoss={running_loss/n_seen:.4f} | trainER={running_reward/n_seen:.4f} | "
        f"valER={val_total/val_n:.4f} | valEntropy={router_ent:.3f}"
    )

# ------------------ TOP-K MIXTURE REWARD ------------------
def topk_mixture_reward(router, loader, k):
    router.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            probs = F.softmax(router(x), dim=1)
            topk_vals, topk_idx = probs.topk(k, dim=1)
            topk_vals = topk_vals / (topk_vals.sum(dim=1, keepdim=True) + 1e-12)
            rewards = y.gather(1, topk_idx)
            total += (topk_vals * rewards).sum(dim=1).sum().item()
            n += x.size(0)
    return total / n

# ------------------ ACCURACY METRICS ------------------
def accuracy_from_rewards(reward_vec):
    return (reward_vec > 0).float().mean().item()

def topk_mixture_accuracy(router, loader, k):
    router.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            probs = F.softmax(router(x), dim=1)
            topk_vals, topk_idx = probs.topk(k, dim=1)
            topk_vals = topk_vals / (topk_vals.sum(dim=1, keepdim=True) + 1e-12)
            rewards = y.gather(1, topk_idx)
            exp_reward = (topk_vals * rewards).sum(dim=1)
            correct += (exp_reward > 0).sum().item()
            total += x.size(0)
    return correct / total

# ------------------ FINAL RESULTS ------------------
val_mix_reward = topk_mixture_reward(router, val_loader, TOPK)
test_mix_reward = topk_mixture_reward(router, test_loader, TOPK)

best_single_val_acc = accuracy_from_rewards(Y_val[:, best_single_idx])
best_single_test_acc = accuracy_from_rewards(Y_test[:, best_single_idx])

router_val_acc = topk_mixture_accuracy(router, val_loader, TOPK)
router_test_acc = topk_mixture_accuracy(router, test_loader, TOPK)

oracle_val_acc = accuracy_from_rewards(Y_val.max(dim=1).values)
oracle_test_acc = accuracy_from_rewards(Y_test.max(dim=1).values)

print("\n=== FINAL EVALUATION ===")
print(f"TOPK = {TOPK}")
print()
print("REWARD")
print(f"  Best single VAL:   {best_single_val_reward:.4f}")
print(f"  Router mix VAL:    {val_mix_reward:.4f}")
print(f"  Oracle VAL:        {oracle_val_reward:.4f}")
print()
print(f"  Best single TEST:  {best_single_test_reward:.4f}")
print(f"  Router mix TEST:   {test_mix_reward:.4f}")
print(f"  Oracle TEST:       {oracle_test_reward:.4f}")
print()
print("ACCURACY (reward > 0)")
print(f"  Best single VAL:   {best_single_val_acc:.2%}")
print(f"  Router mix VAL:    {router_val_acc:.2%}")
print(f"  Oracle VAL:        {oracle_val_acc:.2%}")
print()
print(f"  Best single TEST:  {best_single_test_acc:.2%}")
print(f"  Router mix TEST:   {router_test_acc:.2%}")
print(f"  Oracle TEST:       {oracle_test_acc:.2%}")
print("======================")

# ------------------ ROUTING DISTRIBUTION (TOP-1) ------------------
router.eval()
top1_counts = torch.zeros(num_experts, dtype=torch.long)

with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(DEVICE)
        probs = F.softmax(router(x), dim=1)
        top1 = probs.argmax(dim=1)
        top1_counts.scatter_add_(0, top1.cpu(), torch.ones_like(top1.cpu(), dtype=torch.long))

top1_dist = top1_counts.float() / top1_counts.sum().clamp_min(1)

print("\n=== TEST SET ROUTING DISTRIBUTION (TOP-1) ===")
for i, p in enumerate(top1_dist):
    print(f"Expert {i:02d}: {p:.4%}")
print("============================================\n")

# ------------------ ROUTING ENTROPY (DIAGNOSTIC) ------------------
def routing_entropy(router, loader):
    router.eval()
    ent, n = 0.0, 0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(DEVICE)
            p = F.softmax(router(x), dim=1)
            ent += (-(p * (p.clamp_min(1e-12)).log()).sum(dim=1)).sum().item()
            n += x.size(0)
    return ent / max(1, n)

print(f"Val routing entropy:  {routing_entropy(router, val_loader):.3f}")
print(f"Test routing entropy: {routing_entropy(router, test_loader):.3f}")


Loading data from router_data_checkpoint.pt
Found reward matrix with shape: (17590, 30) (N, num_experts)
>>> Removing duplicate (X, Y) pairs
Deduplicated dataset size: 17590
Removed 0 duplicates

=== DATASET STATS ===
Total datapoints:      17590
Train datapoints:      14072
Validation datapoints: 1759
Test datapoints:       1759
Number of experts:     30


=== BASELINES ===
Best single expert idx:  14
Best single VAL reward:  3.3487
Best single TEST reward: 3.3512
Oracle VAL reward:       5.9287
Oracle TEST reward:      5.8368

>>> Training router (top-K aligned + temperature anneal + entropy schedule + oracle-CE aux + optional hard reweight)
Epoch 00 | tau=2.000 | topk_loss=N | trainLoss=-1.7081 | trainER=2.1191 | valER=2.7646 | valEntropy=1.351
Epoch 01 | tau=1.924 | topk_loss=N | trainLoss=-2.1918 | trainER=2.8377 | valER=2.9803 | valEntropy=1.110
Epoch 02 | tau=1.851 | topk_loss=N | trainLoss=-2.1807 | trainER=2.9242 | valER=2.9565 | valEntropy=1.332
Epoch 03 | tau=1.781 | topk_lo